<a href="https://colab.research.google.com/github/camgenomicmedicine/GMO4/blob/main/Variant_Calling_Workshop_Snakemake_Solution_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Variant calling workshop — Snakemake solution notebook for Google Colab

This is the Colab version of the worked answer-sheet notebook for the variant-calling workshop using the reduced human exome FASTQ file from Bea Rienhoff's data.

You should upload the reads file into `/content` using the Colab file manager before running the setup cells:

```text
daughters_reads.fastq
```

The reads only align to chromosomes 14 and 18. To keep the practical small, the notebook downloads only GRCh38 chromosome 14 and chromosome 18 FASTA files from Ensembl, concatenates them into a small reference genome, and then runs the variant-calling workflow with Snakemake.

Final output for upload to Ensembl VEP:

```text
/content/rienhoff_variant_colab/vcf/daughter.vep_input.vcf
```

This notebook installs all required command-line tools into a Colab-local micromamba environment. Nothing is installed permanently.

## How this differs from the earlier yeast workflow you ran

The broad structure is still familiar:

**FASTQ QC → reference indexing → alignment → BAM sorting/indexing → duplicate marking → variant calling → VCF inspection**

Several earlier workflow steps are simplified or skipped:

| Earlier yeast practical | This answer-sheet workflow |
|---|---|
| Two lanes of paired-end reads | One single-end FASTQ file |
| Yeast reference already provided | Human GRCh38 chromosome 14 + chromosome 18 are downloaded |
| Adapter/quality trimming | Not repeated here; this cut-down teaching file is used directly |
| Lane-level BAM merging | Not needed because there is only one FASTQ file |
| FreeBayes/GATK comparison | Not needed for the answer sheet; we generate a single VCF for VEP |
| IGV exploration | Optional; the main output is a VCF for VEP annotation |

The main teaching point is that the same pipeline logic can be packaged as a reproducible Snakemake workflow.

## 1. Install the required tools

Run this cell once per fresh Colab runtime.

The tools are installed into:

```text
/content/bioinfo_env
```

This can take several minutes the first time it is run.

In [ ]:
%%bash
set -euo pipefail

mkdir -p /content/bin
cd /content

if [ ! -x /content/bin/micromamba ]; then
  echo "Installing micromamba..."
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
else
  echo "micromamba already installed."
fi

if [ ! -x /content/bioinfo_env/bin/snakemake ]; then
  echo "Creating bioinformatics environment..."
  /content/bin/micromamba create -y -p /content/bioinfo_env \
    -c conda-forge -c bioconda \
    python=3.11 \
    snakemake-minimal \
    fastqc \
    bwa \
    samtools \
    bcftools \
    picard \
    gatk4 \
    graphviz \
    wget \
    openjdk
else
  echo "bioinfo_env already exists."
fi

export PATH="/content/bioinfo_env/bin:/content/bin:$PATH"

echo
echo "Tool versions:"
snakemake --version
bwa 2>&1 | sed -n '1,3p' || true
samtools --version | sed -n '1,2p'
bcftools --version | sed -n '1,2p'
gatk --version | sed -n '1,2p' || true
picard MarkDuplicates --version || true
dot -V || true

## 2. Upload and prepare `daughters_reads.fastq`

Upload `daughters_reads.fastq` into `/content` using the Colab file manager on the left.

Then run the cell below. It will copy the FASTQ into a clean working directory:

```text
/content/rienhoff_variant_colab
```

The workflow accepts either `daughters_reads.fastq` or `daughters_reads.fastq.gz`. If a gzipped file is uploaded, it is decompressed into the working directory.

In [ ]:
from pathlib import Path
import os, shutil, gzip

CONTENT = Path("/content")
WORKDIR = CONTENT / "rienhoff_variant_colab"
WORKDIR.mkdir(exist_ok=True)

# Look for the uploaded reads file in /content.
candidates = [
    CONTENT / "daughters_reads.fastq",
    CONTENT / "daughters_reads.fq",
    CONTENT / "daughters_reads.fastq.gz",
    CONTENT / "daughters_reads.fq.gz",
]

source = next((p for p in candidates if p.exists()), None)

# Optional upload prompt if the file is not already present.
if source is None:
    try:
        from google.colab import files
        print("Please upload daughters_reads.fastq or daughters_reads.fastq.gz.")
        uploaded = files.upload()
        uploaded_names = list(uploaded)
        uploaded_candidates = [
            CONTENT / name for name in uploaded_names
            if name.endswith((".fastq", ".fq", ".fastq.gz", ".fq.gz"))
        ]
        if len(uploaded_candidates) == 1:
            source = uploaded_candidates[0]
        elif len(uploaded_candidates) > 1:
            raise FileExistsError(
                "More than one FASTQ-like file was uploaded. Please keep only daughters_reads.fastq in /content."
            )
    except ImportError:
        pass

if source is None or not source.exists():
    raise FileNotFoundError(
        "Could not find daughters_reads.fastq in /content. "
        "Upload the file using the Colab file manager and rerun this cell."
    )

target = WORKDIR / "daughters_reads.fastq"

if source.suffix == ".gz":
    print(f"Decompressing {source.name} to {target} ...")
    with gzip.open(source, "rb") as inp, open(target, "wb") as out:
        shutil.copyfileobj(inp, out)
else:
    print(f"Copying {source.name} to {target} ...")
    shutil.copy2(source, target)

os.environ["WORKDIR"] = str(WORKDIR)
os.chdir(WORKDIR)

print("Working directory:", WORKDIR)
print("Input FASTQ:", target)
print("FASTQ size:", f"{target.stat().st_size / 1_000_000:.2f} MB")

## 3. Check that the command-line tools are available

This checks the tools inside the Colab micromamba environment.

In [ ]:
%%bash
set -euo pipefail
export PATH="/content/bioinfo_env/bin:/content/bin:$PATH"
cd "$WORKDIR"

for tool in wget fastqc bwa samtools picard gatk bcftools snakemake dot; do
  printf "%-12s" "$tool"
  command -v "$tool" || true
done

## 4. Write the Snakefile

The Snakefile does the full analysis:

1. copy the uploaded FASTQ into a `reads/` directory;
2. download GRCh38 chromosome 14 and chromosome 18 FASTA files from Ensembl;
3. concatenate the two chromosomes into a small reference;
4. build the `samtools`, Picard and BWA reference indexes;
5. run FastQC;
6. align the reads with BWA-MEM;
7. sort and index the BAM;
8. mark duplicates with Picard;
9. call variants with GATK HaplotypeCaller;
10. apply a simple QUAL filter;
11. write a VCF for upload to Ensembl VEP.

In [ ]:
from pathlib import Path

snakefile_text = r'''
# Snakefile: Rienhoff chr14+chr18 variant-calling answer sheet for Colab

SAMPLE = "daughter"
FASTQ = "reads/daughters_reads.fastq"

CHROMS = ["14", "18"]
ENSEMBL_FASTA_URL = "https://ftp.ensembl.org/pub/current_fasta/homo_sapiens/dna"

REF = "reference/GRCh38_chr14_chr18.fa"
DICT = "reference/GRCh38_chr14_chr18.dict"
THREADS = 2

rule all:
    input:
        "qc/daughters_reads_fastqc.html",
        REF,
        REF + ".fai",
        DICT,
        REF + ".bwt",
        "bam/daughter.markdup.bam",
        "bam/daughter.markdup.bam.bai",
        "qc/daughter.flagstat.txt",
        "qc/daughter.bcftools_stats.txt",
        "vcf/daughter.raw.vcf",
        "vcf/daughter.filtered.vcf",
        "vcf/daughter.vep_input.vcf"

rule prepare_fastq:
    input:
        "daughters_reads.fastq"
    output:
        FASTQ
    shell:
        "mkdir -p reads && cp {input} {output}"

rule download_reference_chromosomes:
    output:
        expand("reference/Homo_sapiens.GRCh38.dna.chromosome.{chrom}.fa.gz", chrom=CHROMS)
    shell:
        r"""
        mkdir -p reference
        wget -nc -P reference {ENSEMBL_FASTA_URL}/Homo_sapiens.GRCh38.dna.chromosome.14.fa.gz
        wget -nc -P reference {ENSEMBL_FASTA_URL}/Homo_sapiens.GRCh38.dna.chromosome.18.fa.gz
        """

rule concatenate_reference:
    input:
        expand("reference/Homo_sapiens.GRCh38.dna.chromosome.{chrom}.fa.gz", chrom=CHROMS)
    output:
        REF
    shell:
        "zcat {input} > {output}"

rule samtools_faidx:
    input:
        REF
    output:
        REF + ".fai"
    shell:
        "samtools faidx {input}"

rule picard_dictionary:
    input:
        REF
    output:
        DICT
    shell:
        "picard CreateSequenceDictionary R={input} O={output}"

rule bwa_index:
    input:
        REF
    output:
        REF + ".amb",
        REF + ".ann",
        REF + ".bwt",
        REF + ".pac",
        REF + ".sa"
    shell:
        "bwa index {input}"

rule fastqc:
    input:
        FASTQ
    output:
        "qc/daughters_reads_fastqc.html",
        "qc/daughters_reads_fastqc.zip"
    shell:
        "mkdir -p qc && fastqc -o qc {input}"

rule align_and_sort:
    input:
        ref=REF,
        ref_index=REF + ".bwt",
        fastq=FASTQ
    output:
        "bam/daughter.sorted.bam"
    log:
        "logs/bwa_mem.log"
    threads:
        THREADS
    shell:
        r"""
        mkdir -p bam logs
        bwa mem -t {threads} \
          -R '@RG\tID:daughter\tSM:Bea_Rienhoff\tPL:ILLUMINA\tLB:exome\tPU:chr14chr18_subset' \
          {input.ref} {input.fastq} 2> {log} \
        | samtools view -b - \
        | samtools sort -@ {threads} -o {output} -
        """

rule index_sorted_bam:
    input:
        "bam/daughter.sorted.bam"
    output:
        "bam/daughter.sorted.bam.bai"
    shell:
        "samtools index {input} {output}"

rule mark_duplicates:
    input:
        bam="bam/daughter.sorted.bam",
        bai="bam/daughter.sorted.bam.bai"
    output:
        bam="bam/daughter.markdup.bam",
        metrics="qc/daughter.duplicate_metrics.txt"
    shell:
        r"""
        mkdir -p qc
        picard MarkDuplicates \
          INPUT={input.bam} \
          OUTPUT={output.bam} \
          METRICS_FILE={output.metrics} \
          VALIDATION_STRINGENCY=SILENT
        """

rule index_markdup_bam:
    input:
        "bam/daughter.markdup.bam"
    output:
        "bam/daughter.markdup.bam.bai"
    shell:
        "samtools index {input} {output}"

rule flagstat:
    input:
        "bam/daughter.markdup.bam"
    output:
        "qc/daughter.flagstat.txt"
    shell:
        "samtools flagstat {input} > {output}"

rule haplotypecaller:
    input:
        ref=REF,
        fai=REF + ".fai",
        dict=DICT,
        bam="bam/daughter.markdup.bam",
        bai="bam/daughter.markdup.bam.bai"
    output:
        "vcf/daughter.raw.vcf"
    log:
        "logs/gatk_haplotypecaller.log"
    shell:
        r"""
        mkdir -p vcf logs
        gatk HaplotypeCaller \
          -R {input.ref} \
          -I {input.bam} \
          -O {output} \
          2> {log}
        """

rule filter_variants:
    input:
        ref=REF,
        vcf="vcf/daughter.raw.vcf"
    output:
        "vcf/daughter.filtered.vcf"
    log:
        "logs/gatk_variantfiltration.log"
    shell:
        r"""
        gatk VariantFiltration \
          -R {input.ref} \
          -V {input.vcf} \
          -O {output} \
          --filter-name "low_qual" \
          --filter-expression "QUAL < 30.0" \
          2> {log}
        """

rule make_vep_input:
    input:
        "vcf/daughter.filtered.vcf"
    output:
        "vcf/daughter.vep_input.vcf"
    shell:
        r"""
        awk 'BEGIN{{OFS="\t"}} /^#/ || $7=="PASS" || $7=="." {{print}}' {input} > {output}
        """

rule vcf_stats:
    input:
        "vcf/daughter.vep_input.vcf"
    output:
        "qc/daughter.bcftools_stats.txt"
    shell:
        "bcftools stats {input} > {output}"

rule clean:
    shell:
        "rm -rf reads reference qc bam vcf logs .snakemake"
'''
Path("Snakefile").write_text(snakefile_text)
print("Wrote:", Path("Snakefile").resolve())
print("\nFirst 80 lines of the Snakefile:")
print("\n".join(Path("Snakefile").read_text().splitlines()[:80]))

## 5. Dry run

A dry run checks the workflow logic without running any analysis. This is an important Snakemake habit.

In [ ]:
%%bash
set -euo pipefail
export PATH="/content/bioinfo_env/bin:/content/bin:$PATH"
export _JAVA_OPTIONS="-Xmx3g"
cd "$WORKDIR"

snakemake -n --cores 2

## 6. Draw the workflow DAG

The directed acyclic graph shows which files depend on which rules. This is very useful when explaining or debugging a workflow.

In [ ]:
%%bash
set -euo pipefail
export PATH="/content/bioinfo_env/bin:/content/bin:$PATH"
cd "$WORKDIR"

snakemake --dag | dot -Tpng > dag.png
ls -lh dag.png

In [ ]:
from IPython.display import Image, display
from pathlib import Path

dag = Path("dag.png")
if dag.exists():
    display(Image(filename=str(dag)))
else:
    print("dag.png was not created.")

## 7. Run the workflow

This is the main analysis cell.

It downloads the mini-reference and then runs the full pipeline. Rerunning the cell is safe: Snakemake will only rerun jobs whose outputs are missing or out of date.

In [ ]:
%%bash
set -euo pipefail
export PATH="/content/bioinfo_env/bin:/content/bin:$PATH"
export _JAVA_OPTIONS="-Xmx3g"
cd "$WORKDIR"

snakemake --cores 2 --rerun-incomplete

## 8. Inspect the output files

In [ ]:
%%bash
set -euo pipefail
cd "$WORKDIR"

echo "Reference files:"
ls -lh reference | sed -n '1,20p'

echo
echo "BAM files:"
ls -lh bam

echo
echo "VCF files:"
ls -lh vcf

echo
echo "QC files:"
ls -lh qc

## 9. Alignment summary

In [ ]:
%%bash
set -euo pipefail
cd "$WORKDIR"

cat qc/daughter.flagstat.txt

## 10. Count VCF records

In [ ]:
%%bash
set -euo pipefail
cd "$WORKDIR"

for f in vcf/daughter.raw.vcf vcf/daughter.filtered.vcf vcf/daughter.vep_input.vcf; do
    echo "$f"
    awk '!/^#/ {n++} END {print "  variant records:", n+0}' "$f"
done

## 11. Preview the VCF for VEP

Upload this file to the Ensembl Variant Effect Predictor web interface:

```text
vcf/daughter.vep_input.vcf
```

Use the human GRCh38 assembly.

In [ ]:
%%bash
set -euo pipefail
cd "$WORKDIR"

echo "First five non-header VCF records:"
awk '!/^#/ {print; n++; if (n==5) exit}' vcf/daughter.vep_input.vcf

echo
echo "Full path to VEP input VCF:"
readlink -f vcf/daughter.vep_input.vcf

## 12. Download the VCF from Colab

Run this optional cell if you want Colab to download the VCF to your computer for VEP upload.

In [ ]:
from pathlib import Path

vep_vcf = Path(WORKDIR) / "vcf" / "daughter.vep_input.vcf"
print("VCF path:", vep_vcf)

try:
    from google.colab import files
    files.download(str(vep_vcf))
except Exception as exc:
    print("Download did not start automatically:", exc)
    print("Use the file browser to download:", vep_vcf)

## 13. Pull candidate genes for Marfan syndrome and Loeys-Dietz syndrome

The VEP output will tell you which genes overlap the variants. To interpret the answer, we need a candidate-gene list for the relevant clinical differential diagnosis.

This cell downloads HPO `genes_to_disease.txt`, searches for Marfan syndrome, Loeys-Dietz syndrome, and Rienhoff/Reinhoff-related entries, and writes:

```text
candidate_genes_marfan_loeys_dietz.tsv
```

A small fallback list is included in case the live database query fails.

In [ ]:
# @title Code hidden for brevity
import io
import re
import urllib.request
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

os.chdir(WORKDIR)

# HPO stores gene-to-disease associations using disease IDs such as OMIM:212050.
# Therefore we download TWO files:
#   1. genes_to_disease.txt: gene_symbol <-> disease_id
#   2. phenotype.hpoa: disease_id <-> disease_name
# We then join them before searching for disease names.

URLS = {
    "genes_to_disease": [
        "https://github.com/obophenotype/human-phenotype-ontology/releases/latest/download/genes_to_disease.txt",
        "https://purl.obolibrary.org/obo/hp/hpoa/genes_to_disease.txt",
    ],
    "phenotype_hpoa": [
        "https://github.com/obophenotype/human-phenotype-ontology/releases/latest/download/phenotype.hpoa",
        "https://purl.obolibrary.org/obo/hp/phenotype.hpoa",
    ],
}


def download_first_available(label, urls, timeout=90):
    """Try each URL in turn and return decoded text from the first one that works."""
    last_error = None
    for url in urls:
        try:
            print(f"Downloading {label}: {url}")
            with urllib.request.urlopen(url, timeout=timeout) as handle:
                return handle.read().decode("utf-8")
        except Exception as exc:
            print(f"  failed: {exc}")
            last_error = exc
    raise RuntimeError(f"Could not download {label}. Last error: {last_error}")


def normalise_colname(name):
    return re.sub(r"[^a-z0-9]+", "_", str(name).lower()).strip("_")


def read_genes_to_disease(text):
    """Read HPO genes_to_disease.txt."""
    lines = [line for line in text.splitlines() if line.strip() and not line.startswith("#")]
    if not lines:
        return pd.DataFrame()

    df = pd.read_csv(io.StringIO("\n".join(lines)), sep="\t", dtype=str)
    df = df.rename(columns={c: normalise_colname(c) for c in df.columns})

    expected = {"ncbi_gene_id", "gene_symbol", "association_type", "disease_id", "source"}
    missing = expected - set(df.columns)
    if missing:
        raise ValueError(
            "genes_to_disease.txt did not have the expected columns. "
            f"Missing: {sorted(missing)}. Found: {list(df.columns)}"
        )

    df["disease_id"] = df["disease_id"].astype(str).str.strip()
    df["gene_symbol"] = df["gene_symbol"].astype(str).str.strip()
    return df


def read_phenotype_hpoa(text):
    """Read HPO phenotype.hpoa and return disease ID to disease name mapping."""
    header = None
    data_lines = []

    for raw_line in text.splitlines():
        line = raw_line.rstrip("\n")
        if not line.strip():
            continue

        # phenotype.hpoa can contain metadata comment lines. In some releases,
        # the column header itself may be prefixed with '#'. Keep it if found.
        if line.startswith("#"):
            possible_header = line.lstrip("#").strip()
            if possible_header.lower().startswith(("databaseid\t", "database_id\t")):
                header = possible_header.split("\t")
            continue

        fields = line.split("\t")
        if header is None and fields[0].lower() in {"databaseid", "database_id"}:
            header = fields
            continue

        data_lines.append(line)

    if not data_lines:
        return pd.DataFrame(columns=["disease_id", "disease_name"])

    if header is None:
        # Current phenotype.hpoa format begins with these columns.
        header = [
            "DatabaseID", "DiseaseName", "Qualifier", "HPO_ID", "Reference",
            "Evidence", "Onset", "Frequency", "Sex", "Modifier", "Aspect", "Biocuration"
        ]

    max_cols = max(len(line.split("\t")) for line in data_lines)
    header = header[:max_cols] + [f"extra_{i}" for i in range(len(header), max_cols)]

    hpoa = pd.read_csv(
        io.StringIO("\n".join(data_lines)),
        sep="\t",
        names=header,
        dtype=str,
        engine="python",
    )
    hpoa = hpoa.rename(columns={c: normalise_colname(c) for c in hpoa.columns})

    # Handle common variants of the column names.
    id_col = next((c for c in hpoa.columns if c in {"databaseid", "database_id", "disease_id"}), None)
    name_col = next((c for c in hpoa.columns if c in {"diseasename", "disease_name"}), None)

    if id_col is None or name_col is None:
        raise ValueError(
            "Could not identify disease ID/name columns in phenotype.hpoa. "
            f"Found columns: {list(hpoa.columns)}"
        )

    disease_map = (
        hpoa[[id_col, name_col]]
        .rename(columns={id_col: "disease_id", name_col: "disease_name"})
        .dropna()
        .drop_duplicates()
    )
    disease_map["disease_id"] = disease_map["disease_id"].astype(str).str.strip()
    disease_map["disease_name"] = disease_map["disease_name"].astype(str).str.strip()
    return disease_map


try:
    genes_text = download_first_available("genes_to_disease.txt", URLS["genes_to_disease"])
    hpoa_text = download_first_available("phenotype.hpoa", URLS["phenotype_hpoa"])

    genes_to_disease = read_genes_to_disease(genes_text)
    disease_map = read_phenotype_hpoa(hpoa_text)

    print(f"genes_to_disease rows: {len(genes_to_disease):,}")
    print(f"disease ID/name rows: {len(disease_map):,}")

    genes_with_names = genes_to_disease.merge(
        disease_map,
        on="disease_id",
        how="left",
    )

    query_patterns = {
        "Marfan syndrome": r"\bMarfan\b",
        "Loeys-Dietz syndrome": r"Loeys[\s\-–]?Dietz|Dietz syndrome",
        "Rienhoff/Reinhoff-related syndrome": r"Rienhoff|Reinhoff",
    }

    disease_hits = []
    gene_hits = []

    for label, pattern in query_patterns.items():
        matched_diseases = disease_map[
            disease_map["disease_name"].str.contains(pattern, case=False, regex=True, na=False)
        ].copy()

        if not matched_diseases.empty:
            matched_diseases["query_condition"] = label
            disease_hits.append(matched_diseases)

            matched_genes = genes_with_names[
                genes_with_names["disease_id"].isin(matched_diseases["disease_id"])
            ].copy()

            if not matched_genes.empty:
                matched_genes["query_condition"] = label
                gene_hits.append(matched_genes)

    if disease_hits:
        matched_diseases = (
            pd.concat(disease_hits, ignore_index=True)
            [["query_condition", "disease_id", "disease_name"]]
            .drop_duplicates()
            .sort_values(["query_condition", "disease_name", "disease_id"])
        )
    else:
        matched_diseases = pd.DataFrame(columns=["query_condition", "disease_id", "disease_name"])

    if gene_hits:
        candidates = pd.concat(gene_hits, ignore_index=True)
        candidates = (
            candidates[[
                "query_condition", "gene_symbol", "ncbi_gene_id", "association_type",
                "disease_id", "disease_name", "source"
            ]]
            .drop_duplicates()
            .sort_values(["query_condition", "gene_symbol", "disease_name"])
        )
    else:
        candidates = pd.DataFrame()

except Exception as exc:
    print("Online HPO query failed:", exc)
    matched_diseases = pd.DataFrame()
    candidates = pd.DataFrame()

# Teaching fallback: keep the practical usable if the live HPO query changes or is unavailable.
if candidates.empty:
    candidates = pd.DataFrame({
        "query_condition": [
            "fallback", "fallback", "fallback", "fallback",
            "fallback", "fallback", "fallback", "fallback", "fallback"
        ],
        "gene_symbol": ["FBN1", "TGFBR1", "TGFBR2", "SMAD2", "SMAD3", "TGFB2", "TGFB3", "SKI", "IPO8"],
        "ncbi_gene_id": [pd.NA] * 9,
        "association_type": ["fallback"] * 9,
        "disease_id": [pd.NA] * 9,
        "disease_name": ["Fallback list: live HPO query returned no gene matches"] * 9,
        "source": [pd.NA] * 9,
    })

matched_diseases_out = Path("candidate_disease_matches_marfan_loeys_dietz.tsv")
candidate_genes_out = Path("candidate_genes_marfan_loeys_dietz.tsv")

if not matched_diseases.empty:
    matched_diseases.to_csv(matched_diseases_out, sep="\t", index=False)
    print("Saved matched disease IDs:", matched_diseases_out.resolve())

candidates.to_csv(candidate_genes_out, sep="\t", index=False)
print("Saved candidate genes:", candidate_genes_out.resolve())

print("\nUnique candidate genes:")
print(", ".join(sorted(candidates["gene_symbol"].dropna().unique())))

print("\nMatched diseases used for the lookup:")
display(
    matched_diseases
    if not matched_diseases.empty
    else pd.DataFrame({"message": ["No live disease-name matches found; fallback genes used."]})
)

print("\nCandidate genes:")
display(candidates)

## 14. Compare a downloaded VEP result with the candidate genes

After running VEP online, download the result as a tab-delimited text file and upload it back into this Colab working directory as:

```text
/content/rienhoff_variant_colab/vep_output.txt
```

Then run the cell below.

In [ ]:
# @title Code hidden for brevity
from pathlib import Path
import json
import os
import time
import urllib.parse
import urllib.request

import pandas as pd
from IPython.display import display

# Colab notebook sets WORKDIR globally. In the VM notebook, WORKDIR is also set earlier.
os.chdir(WORKDIR)

def read_vep_tabular(path):
    """Read a VEP tab-delimited output file.

    VEP tabular output usually has metadata lines beginning with ## and a
    header line beginning with #. This function keeps the real header.
    """
    header = None
    rows = []

    with open(path) as handle:
        for line in handle:
            line = line.rstrip("\n")
            if not line:
                continue
            if line.startswith("##"):
                continue
            if line.startswith("#"):
                header = line.lstrip("#").split("\t")
                continue
            rows.append(line.split("\t"))

    if header is None:
        return pd.read_csv(path, sep="\t", comment="#", dtype=str)

    # Some VEP exports contain optional extra columns. Pad or trim rows defensively.
    fixed_rows = []
    for row in rows:
        if len(row) < len(header):
            row = row + [""] * (len(header) - len(row))
        elif len(row) > len(header):
            row = row[:len(header)]
        fixed_rows.append(row)

    return pd.DataFrame(fixed_rows, columns=header)


def strip_ensembl_version(value):
    """Remove version suffixes such as ENSG000001234.5 -> ENSG000001234."""
    if pd.isna(value):
        return ""
    return str(value).strip().split(".")[0]


# Fallback IDs are included so that the practical remains usable if the live
# Ensembl REST lookup is unavailable during a teaching session.
FALLBACK_ENSEMBL_IDS = {
    "FBN1": ["ENSG00000166147"],
    "TGFBR1": ["ENSG00000106799"],
    "TGFBR2": ["ENSG00000163513"],
    "SMAD2": ["ENSG00000175387"],
    "SMAD3": ["ENSG00000166949"],
    "TGFB2": ["ENSG00000092969"],
    "TGFB3": ["ENSG00000119699"],
    "SKI": ["ENSG00000157933"],
    "IPO8": ["ENSG00000148308"],
}


def lookup_ensembl_gene_ids(symbol, timeout=20):
    """Return Ensembl gene IDs for a human HGNC gene symbol using Ensembl REST."""
    quoted = urllib.parse.quote(str(symbol))
    url = (
        "https://rest.ensembl.org/xrefs/symbol/homo_sapiens/"
        f"{quoted}?object_type=gene"
    )

    request = urllib.request.Request(
        url,
        headers={
            "Accept": "application/json",
            "Content-Type": "application/json",
        },
    )

    try:
        with urllib.request.urlopen(request, timeout=timeout) as handle:
            records = json.loads(handle.read().decode("utf-8"))
    except Exception as exc:
        print(f"Ensembl lookup failed for {symbol}: {exc}")
        return FALLBACK_ENSEMBL_IDS.get(str(symbol), [])

    ids = sorted({
        strip_ensembl_version(record.get("id", ""))
        for record in records
        if str(record.get("id", "")).startswith("ENSG")
    })

    # Avoid hammering the public REST API when looking up several symbols.
    time.sleep(0.1)

    if not ids:
        ids = FALLBACK_ENSEMBL_IDS.get(str(symbol), [])

    return ids


vep_path = Path("vep_output.txt")
candidate_path = Path("candidate_genes_marfan_loeys_dietz.tsv")

if not candidate_path.exists():
    raise FileNotFoundError(
        "Could not find candidate_genes_marfan_loeys_dietz.tsv. "
        "Run the previous candidate-gene cell first."
    )

candidates = pd.read_csv(candidate_path, sep="\t", dtype=str)
if "gene_symbol" not in candidates.columns:
    raise ValueError(
        "candidate_genes_marfan_loeys_dietz.tsv does not contain a gene_symbol column."
    )

candidate_symbols = sorted(candidates["gene_symbol"].dropna().astype(str).unique())

print("Mapping candidate gene symbols to Ensembl gene IDs...")
symbol_to_ids = {
    symbol: lookup_ensembl_gene_ids(symbol)
    for symbol in candidate_symbols
}

mapping_rows = []
for symbol, ensembl_ids in symbol_to_ids.items():
    if not ensembl_ids:
        mapping_rows.append({"gene_symbol": symbol, "ensembl_gene_id": pd.NA})
    else:
        for ensembl_id in ensembl_ids:
            mapping_rows.append({"gene_symbol": symbol, "ensembl_gene_id": ensembl_id})

candidate_id_map = pd.DataFrame(mapping_rows).drop_duplicates()

# Join the Ensembl IDs back onto the candidate table for interpretability.
candidates_with_ids = candidates.merge(
    candidate_id_map,
    on="gene_symbol",
    how="left",
)

candidate_id_map.to_csv(
    "candidate_gene_symbol_to_ensembl.tsv",
    sep="\t",
    index=False,
)
candidates_with_ids.to_csv(
    "candidate_genes_marfan_loeys_dietz_with_ensembl_ids.tsv",
    sep="\t",
    index=False,
)

print("\nCandidate genes with Ensembl IDs:")
display(
    candidates_with_ids[
        ["gene_symbol", "ensembl_gene_id", "query_condition", "disease_id", "disease_name"]
    ].drop_duplicates().sort_values(["gene_symbol", "ensembl_gene_id"])
)

if not vep_path.exists():
    print("\nNo vep_output.txt file found yet.")
    print("Download the tabular VEP result and save/upload it here:")
    print((Path(WORKDIR) / "vep_output.txt").resolve())
else:
    vep = read_vep_tabular(vep_path)

    if "Gene" not in vep.columns:
        print("No Gene column found in the VEP output.")
        print("Available columns:", list(vep.columns))
        raise ValueError(
            "The VEP output needs the standard tabular Gene column, which contains Ensembl gene IDs."
        )

    vep = vep.copy()
    vep["Gene_no_version"] = vep["Gene"].apply(strip_ensembl_version)

    candidate_ensembl_ids = set(
        candidate_id_map["ensembl_gene_id"]
        .dropna()
        .astype(str)
        .map(strip_ensembl_version)
    )

    observed_candidate_ids = sorted(
        set(vep["Gene_no_version"].dropna().astype(str)) & candidate_ensembl_ids
    )

    print("\nCandidate Ensembl gene IDs seen in the VEP output:")
    print(", ".join(observed_candidate_ids) if observed_candidate_ids else "No overlap found.")

    if observed_candidate_ids:
        hit_map = (
            candidate_id_map
            .assign(ensembl_gene_id_no_version=lambda df: df["ensembl_gene_id"].map(strip_ensembl_version))
            .query("ensembl_gene_id_no_version in @observed_candidate_ids")
            [["gene_symbol", "ensembl_gene_id"]]
            .drop_duplicates()
        )

        vep_hits = vep[vep["Gene_no_version"].isin(observed_candidate_ids)].copy()
        hit_map = hit_map.assign(
            ensembl_gene_id_no_version=hit_map["ensembl_gene_id"].map(strip_ensembl_version)
        )

        vep_hits = vep_hits.merge(
            hit_map,
            left_on="Gene_no_version",
            right_on="ensembl_gene_id_no_version",
            how="left",
        )

        vep_hits = vep_hits.drop(columns=["ensembl_gene_id_no_version"], errors="ignore")

        print("\nVEP rows overlapping candidate genes:")
        display(vep_hits.drop_duplicates())

        vep_hits.to_csv("vep_candidate_gene_hits.tsv", sep="\t", index=False)
        print("\nSaved overlapping VEP rows to:")
        print((Path(WORKDIR) / "vep_candidate_gene_hits.tsv").resolve())


## 15. Optional clean-up

This removes the workflow outputs but keeps the uploaded FASTQ in the top level of the working directory.

Only run this if you want to start the workflow again from scratch.

In [ ]:
%%bash
set -euo pipefail
export PATH="/content/bioinfo_env/bin:/content/bin:$PATH"
cd "$WORKDIR"

# Uncomment the next line to clean the workflow outputs.
# snakemake clean